In [1]:
DATA_ROOT = "/kaggle/input/140k-real-and-fake-faces/real_vs_fake/real-vs-fake"
!ls -1 $DATA_ROOT; ls -1 $DATA_ROOT/train; ls -1 $DATA_ROOT/valid; ls -1 $DATA_ROOT/test


test
train
valid
fake
real
fake
real
fake
real


In [2]:
# === Stage-2B inline model & dataset (SLIC-enabled, CMF + FFT + Superpixel tokens) ===
import torch, torch.nn as nn, torch.nn.functional as F
import os, sys, time, math, shutil, tempfile, traceback, argparse
from pathlib import Path
from datetime import datetime
import numpy as np
from torchvision.models import resnet50, ResNet50_Weights
from torchvision import transforms
from torchvision.datasets import ImageFolder
from skimage.segmentation import slic

from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, roc_curve
from scipy.optimize import brentq
from scipy.interpolate import interp1d

def calc_metrics(y_true, y_prob):
    y_prob = np.nan_to_num(y_prob, nan=0.5)  
    y_pred = (y_prob > 0.5).astype(int)
    auc  = roc_auc_score(y_true, y_prob)
    f1   = f1_score(y_true, y_pred)
    acc  = accuracy_score(y_true, y_pred)
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    eer  = brentq(lambda x: 1. - x - interp1d(fpr, tpr)(x), 0., 1.)
    return auc, f1, eer, acc


# ---------------- Dataset with SLIC ----------------
from PIL import Image
from io import BytesIO
import random

def random_jpeg(img):
    buf = BytesIO()
    img.save(buf, format="JPEG", quality=random.randint(40, 90))
    buf.seek(0)
    return Image.open(buf).convert("RGB")

class SuperpixelImageFolder(ImageFolder):
    def __init__(self, root, n_segments=25, compactness=10.0, image_size=224):
        tfm = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(0.2,0.2,0.1,0.05),
            transforms.RandomApply([transforms.Lambda(random_jpeg)], p=0.3),
            transforms.ToTensor(),
            transforms.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
        ])
        super().__init__(root, transform=tfm)
        self.n_segments = int(n_segments)
        self.compactness = float(compactness)
        self.image_size = int(image_size)
        self.mean = torch.tensor([0.485,0.456,0.406])[:,None,None]
        self.std  = torch.tensor([0.229,0.224,0.225])[:,None,None]

    def __getitem__(self, idx):
        x_norm, y = super().__getitem__(idx)             # (3,H,W) normalized
        # de-normalize to [0,1] for SLIC
        x_vis = (x_norm*self.std + self.mean).clamp(0,1).permute(1,2,0).cpu().numpy()
        seg = slic(
            x_vis, n_segments=self.n_segments, compactness=self.compactness,
            start_label=0, channel_axis=-1
        ).astype(np.int32)                               # (H,W)
        return x_norm, torch.from_numpy(seg), y


# ---------------- Simple CMF block on feature maps ----------------
class CMF(nn.Module):
    """
    Cross-Modal Fusion on feature maps:
    - project RGB feat map and FFT feat map to d_model
    - Q from RGB, K/V from FFT
    - residual back to RGB channel space (C_in)
    """
    def __init__(self, c_in=1024, d_model=256, heads=4, dropout=0.0):
        super().__init__()
        self.rgb_proj  = nn.Conv2d(c_in, d_model, 1, bias=False)
        self.freq_proj = nn.Conv2d(c_in, d_model, 1, bias=False)
        self.attn      = nn.MultiheadAttention(d_model, heads, batch_first=True, dropout=dropout)
        self.out       = nn.Conv2d(d_model, c_in, 1, bias=False)
        self.out_bn    = nn.BatchNorm2d(c_in)

    def forward(self, rgb_map, freq_map):
        B,C,H,W = rgb_map.shape
        q = self.rgb_proj(rgb_map).flatten(2).transpose(1,2)   # (B,HW,d)
        k = self.freq_proj(freq_map).flatten(2).transpose(1,2) # (B,HW,d)
        v = k
        fused,_ = self.attn(q,k,v)                             # (B,HW,d)
        fused = fused.transpose(1,2).reshape(B,-1,H,W)
        fused = self.out_bn(self.out(fused))
        return rgb_map + fused

# ---------------- Tiny transformer over tokens ----------------
class TransformerBlock(nn.Module):
    def __init__(self, dim, heads=4, mlp_ratio=4.0, dropout=0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn  = nn.MultiheadAttention(dim, heads, batch_first=True, dropout=dropout)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp   = nn.Sequential(
            nn.Linear(dim, int(dim*mlp_ratio)), nn.GELU(),
            nn.Linear(int(dim*mlp_ratio), dim)
        )
    def forward(self, x):
        x = x + self.attn(self.norm1(x), self.norm1(x), self.norm1(x))[0]
        x = x + self.mlp(self.norm2(x))
        return x

# ---------------- Superpixel token pooling ----------------
def superpixel_tokens(feat_map, seg_ids):
    """
    feat_map: (B,C,Hf,Wf)
    seg_ids : (B,H,W) integer ids; will be resized to (Hf,Wf) with nearest
    Returns:
      toks: (B,R,C) padded to max R in batch
      pad_mask: (B,R) True where padded
    """
    B,C,Hf,Wf = feat_map.shape
    seg = F.interpolate(seg_ids.unsqueeze(1).float(), size=(Hf,Wf), mode='nearest').squeeze(1).long()
    toks_list, counts = [], []
    for b in range(B):
        ids = torch.unique(seg[b])
        mapp = feat_map[b]          # (C,Hf,Wf)
        tokens_b = []
        for rid in ids:
            m = (seg[b]==rid).float()
            w = m / (m.sum()+1e-6)
            tok = (mapp * w).sum(dim=(1,2))   # (C,)
            tokens_b.append(tok)
        tokens_b = torch.stack(tokens_b,0)     # (Rb,C)
        toks_list.append(tokens_b)
        counts.append(tokens_b.size(0))
    maxR = max(counts)
    padded = []
    for t in toks_list:
        if t.size(0) < maxR:
            pad = torch.zeros(maxR - t.size(0), t.size(1), device=t.device, dtype=t.dtype)
            t = torch.cat([t, pad], dim=0)
        padded.append(t)
    toks = torch.stack(padded, 0)              # (B,R,C)
    pad_mask = torch.arange(maxR, device=toks.device)[None,:] >= torch.tensor(counts, device=toks.device)[:,None]
    return toks, pad_mask  # pad_mask=True means "ignore"

# ---------------- FFT magnitude map from input ----------------
@torch.no_grad()
def fft_mag_map(x_norm):
    """
    x_norm: (B,3,H,W) normalized; returns single-channel mag map ~ (B,1,H, W//2+1)
    """
    mean = torch.tensor([0.485,0.456,0.406], device=x_norm.device).view(1,3,1,1)
    std  = torch.tensor([0.229,0.224,0.225], device=x_norm.device).view(1,3,1,1)
    x = (x_norm*std + mean).clamp(0,1)
    y = 0.2989*x[:,0] + 0.5870*x[:,1] + 0.1140*x[:,2]   # (B,H,W)
    Y = torch.fft.rfft2(y, norm='ortho')
    mag = torch.log1p(torch.abs(Y)).unsqueeze(1)        # (B,1,H,W//2+1)
    return mag

# ---------------- Full Stage-2B model (SLIC tokens) ----------------
class Stage2(nn.Module):
    def __init__(self, n_segments=25, imagenet_backbone=False,
                 enc_dim=1024, d_model=256, heads=4, enc_layers=2, num_classes=2):
        super().__init__()
        weights = ResNet50_Weights.IMAGENET1K_V1 if imagenet_backbone else None
        resnet = resnet50(weights=weights)
        self.backbone = nn.Sequential(
            resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool,
            resnet.layer1, resnet.layer2, resnet.layer3, resnet.layer4
        )
        self.neck = nn.Sequential(nn.Conv2d(2048, enc_dim, 1, bias=False),
                                  nn.BatchNorm2d(enc_dim), nn.ReLU(inplace=True))
        # project FFT mag (1ch) to enc_dim
        self.fft_proj = nn.Sequential(nn.Conv2d(1, enc_dim, 1, bias=False),
                                      nn.BatchNorm2d(enc_dim), nn.ReLU(inplace=True))
        self.cmf = CMF(c_in=enc_dim, d_model=d_model, heads=heads, dropout=0.0)

        self.encoder = nn.Sequential(*[TransformerBlock(enc_dim, heads=heads) for _ in range(enc_layers)])
        self.head = nn.Sequential(nn.LayerNorm(enc_dim),
                                  nn.Linear(enc_dim, num_classes))

    def forward(self, x, seg_ids):
        # backbone feats
        f = self.backbone(x)           # (B,2048,Hf,Wf)
        f = self.neck(f)               # (B,1024,Hf,Wf)

        # FFT branch on input, resize to feature map, project
        mag = fft_mag_map(x)                                         # (B,1,H,W//2+1)
        mag = F.interpolate(mag, size=f.shape[-2:], mode='bilinear', align_corners=False)
        f_freq = self.fft_proj(mag)                                  # (B,1024,Hf,Wf)

        # CMF residual fusion on maps
        f_fused = self.cmf(f, f_freq)                                # (B,1024,Hf,Wf)

        # Superpixel tokenization
        toks, pad_mask = superpixel_tokens(f_fused, seg_ids)         # (B,R,1024), (B,R)

        # Tiny transformer over tokens
        z = self.encoder(toks)                                       # (B,R,1024)

        # Global mean over valid tokens only
        valid = (~pad_mask).float()                                  # (B,R)
        denom = valid.sum(dim=1, keepdim=True).clamp_min(1.0)        # (B,1)
        pooled = (z * valid.unsqueeze(-1)).sum(dim=1) / denom        # (B,1024)

        logits = self.head(pooled)                                   # (B,2)
        return logits


In [3]:
# === Stage-2B: Imports (patched) ===
import os, sys, time, math, shutil, tempfile, traceback
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader

# If your notebook defines them inline, keep that cell as-is and *do not* import here.

# Optional but recommended for slightly faster dataloaders on Kaggle
torch.backends.cudnn.benchmark = True


In [4]:
# === Stage-2B: Utils (patched) ===

def atomic_save(state, path: str):
    """Robust save that won't leave corrupt files on crash."""
    p = Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    fd, tmp_path = tempfile.mkstemp(dir=str(p.parent))
    os.close(fd)
    torch.save(state, tmp_path)
    os.replace(tmp_path, str(p))

def zip_ckpts():
    """Zip all stage2 checkpoints to a single archive for easy download."""
    os.system("zip -j -q /kaggle/working/stage2_ckpts.zip /kaggle/working/stage2*.pth 2>/dev/null || true")

class AverageMeter:
    def __init__(self):
        self.reset()
    def reset(self):
        self.sum = 0.0
        self.count = 0
    @property
    def avg(self):
        return self.sum / max(1, self.count)
    def update(self, val, n=1):
        self.sum += float(val) * n
        self.count += n

@torch.no_grad()
def top1_accuracy(logits, y):
    return (logits.argmax(1) == y).float().mean().item()


In [5]:
# === Stage-2B Train/Eval (SLIC aware) ===
@torch.no_grad()
def evaluate_with_metrics(model, loader, device, real_idx=1):
    model.eval()
    y_true = []
    y_prob = []
    tot_loss = 0.0
    n = 0
    criterion = nn.CrossEntropyLoss()

    for x, seg, y in loader:
        x, seg, y = x.to(device, non_blocking=True), seg.to(device), y.to(device)
        logits = model(x, seg)

        loss = criterion(logits, y)
        b = x.size(0)
        tot_loss += loss.item() * b
        n += b

        probs = torch.softmax(logits, dim=1)[:, real_idx].detach().cpu().numpy()
        y_true.extend(y.detach().cpu().numpy())
        y_prob.extend(probs)

    y_true = np.array(y_true)
    y_prob = np.array(y_prob)
    auc, f1, eer, acc = calc_metrics(y_true, y_prob)
    return tot_loss / max(1, n), auc, f1, eer, acc


def train_one_epoch(model, criterion, optimizer, loader, device, scaler=None, max_grad_norm=None):
    model.train()
    totL = totA = n = 0

    use_amp = scaler is not None and device.type == "cuda"

    for x, seg, y in loader:
        x, seg, y = x.to(device, non_blocking=True), seg.to(device), y.to(device)

        optimizer.zero_grad(set_to_none=True)

        if use_amp:
            with torch.amp.autocast("cuda"):
                logits = model(x, seg)
                loss = criterion(logits, y)
            scaler.scale(loss).backward()
            if max_grad_norm:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(x, seg)
            loss = criterion(logits, y)
            loss.backward()
            if max_grad_norm:
                nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            optimizer.step()

        b = x.size(0)
        totL += loss.item() * b
        totA += (logits.argmax(1) == y).float().sum().item()
        n += b

    return totL/max(1,n), totA/max(1,n)


In [6]:
def main():
    p = argparse.ArgumentParser()
    p.add_argument('--data_root', type=str, required=True)
    p.add_argument('--save', type=str, default='/kaggle/working/stage2.pth')
    p.add_argument('--ckpt', type=str, default=None)
    p.add_argument('--epochs', type=int, default=6)
    p.add_argument('--batch_size', type=int, default=16)
    p.add_argument('--lr', type=float, default=1e-4)
    p.add_argument('--weight_decay', type=float, default=1e-5)
    p.add_argument('--max_grad_norm', type=float, default=1.0)
    p.add_argument('--n_segments', type=int, default=25)
    p.add_argument('--imagenet_backbone', action='store_true')
    p.add_argument('--image_size', type=int, default=224)
    p.add_argument('--num_workers', type=int, default=4)
    p.add_argument('--pin_memory', action='store_true')
    args = p.parse_args()

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"[{datetime.now()}] device: {device}")

    train_set = SuperpixelImageFolder(
        root=os.path.join(args.data_root, 'train'),
        n_segments=args.n_segments, image_size=args.image_size
    )
    val_set = SuperpixelImageFolder(
        root=os.path.join(args.data_root, 'valid'),
        n_segments=args.n_segments, image_size=args.image_size
    )

    train_loader = DataLoader(train_set, batch_size=args.batch_size, shuffle=True,
                              num_workers=args.num_workers, pin_memory=args.pin_memory, drop_last=True)
    val_loader   = DataLoader(val_set, batch_size=max(8, args.batch_size), shuffle=False,
                              num_workers=args.num_workers, pin_memory=args.pin_memory)

    model = Stage2(
        n_segments=args.n_segments,
        imagenet_backbone=args.imagenet_backbone
    ).to(device)

    if args.ckpt and os.path.isfile(args.ckpt):
        print(f"Loading init checkpoint: {args.ckpt}")
        state = torch.load(args.ckpt, map_location='cpu')
        model.load_state_dict(state, strict=False)

    for p in model.parameters():
        p.requires_grad = True

    n_total = sum(p.numel() for p in model.parameters())
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Trainable params: {n_train:,} / {n_total:,}")

    criterion = nn.CrossEntropyLoss()
    optimizer  = optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
    scaler     = torch.amp.GradScaler("cuda") if device.type == "cuda" else None

    real_idx    = train_set.class_to_idx.get("real", 1)
    best_val_auc = -1.0   # ← outside the loop, fixed

    try:
        for epoch in range(1, args.epochs + 1):
            print(f"\n=== Epoch {epoch}/{args.epochs} ===")
            tr_loss, tr_acc = train_one_epoch(
                model, criterion, optimizer, train_loader, device,
                scaler=scaler, max_grad_norm=args.max_grad_norm
            )
            va_loss, auc, f1, eer, acc = evaluate_with_metrics(
                model, val_loader, device, real_idx=real_idx
            )
            print(f"train: loss={tr_loss:.4f} acc={tr_acc:.4f} | "
                  f"val: loss={va_loss:.4f} | AUROC={auc:.3f} | F1={f1:.3f} | EER={eer:.3f} | ACC={acc:.3f}")

            atomic_save(model.state_dict(), "/kaggle/working/stage2_last.pth")

            if auc > best_val_auc:
                best_val_auc = auc
                atomic_save(model.state_dict(), args.save)
                print(f"[BEST] saved -> {args.save} (val_auc={best_val_auc:.4f})")

            zip_ckpts()

    except KeyboardInterrupt:
        print("Interrupted — saving last checkpoint.")
        atomic_save(model.state_dict(), "/kaggle/working/stage2_last.pth")
        zip_ckpts()
        raise
    except Exception as e:
        print("Exception during training:\n", traceback.format_exc())
        atomic_save(model.state_dict(), "/kaggle/working/stage2_last.pth")
        zip_ckpts()
        raise
    finally:
        os.system("ls -lh /kaggle/working | egrep 'stage2.*\\.pth|stage2_ckpts\\.zip' || true")

In [7]:
# === Stage-2B: Notebook Run (no here-doc, no runpy) ===
import sys
argv = [
  "run",
  "--data_root", "/kaggle/input/140k-real-and-fake-faces/real_vs_fake/real-vs-fake",
  "--epochs", "6",
  "--batch_size", "16",
  "--lr", "1e-4",
  "--n_segments", "25",
  "--image_size", "224",
  "--pin_memory",
  "--imagenet_backbone",  
  "--save", "/kaggle/working/stage2.pth",
]


_argv_bak = sys.argv
try:
    sys.argv = argv
    main()    
finally:
    sys.argv = _argv_bak


[2026-04-21 04:35:48.505590] device: cuda


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100%|██████████| 97.8M/97.8M [00:00<00:00, 193MB/s]


Trainable params: 51,858,498 / 51,858,498

=== Epoch 1/6 ===
train: loss=0.2426 acc=0.9116 | val: loss=0.1868 | AUROC=0.990 | F1=0.930 | EER=0.051 | ACC=0.933
[BEST] saved -> /kaggle/working/stage2.pth (val_auc=0.9901)

=== Epoch 2/6 ===
train: loss=0.1414 acc=0.9575 | val: loss=0.0873 | AUROC=0.996 | F1=0.972 | EER=0.026 | ACC=0.973
[BEST] saved -> /kaggle/working/stage2.pth (val_auc=0.9961)

=== Epoch 3/6 ===
train: loss=0.0936 acc=0.9736 | val: loss=0.0751 | AUROC=0.997 | F1=0.977 | EER=0.023 | ACC=0.977
[BEST] saved -> /kaggle/working/stage2.pth (val_auc=0.9971)

=== Epoch 4/6 ===
train: loss=0.0717 acc=0.9806 | val: loss=0.0590 | AUROC=0.999 | F1=0.982 | EER=0.016 | ACC=0.982
[BEST] saved -> /kaggle/working/stage2.pth (val_auc=0.9987)

=== Epoch 5/6 ===
train: loss=nan acc=0.8420 | val: loss=nan | AUROC=0.500 | F1=0.000 | EER=0.500 | ACC=0.500

=== Epoch 6/6 ===
train: loss=nan acc=0.5000 | val: loss=nan | AUROC=0.500 | F1=0.000 | EER=0.500 | ACC=0.500
-rw-r--r-- 1 root root 184M 

## Optional: Skip Training and Evaluate a Provided Checkpoint

If you do not want to rerun training, you may instead upload the provided Stage 2 checkpoint as a Kaggle dataset and evaluate it directly.

In that case:
- do not run the training cell above(comment out cells)
- update the checkpoint path in the evaluation-only cell below(uncomment cell below)
- run only the evaluation-only cell

Important: use the best saved checkpoint (`stage2.pth`), not the last checkpoint after training (`stage2_last.pth`).


In [ ]:
# # ============================================
# # Optional evaluation-only cell for Stage 2
# # ============================================

# import os
# import torch
# from torch.utils.data import DataLoader

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# CKPT_PATH = "/kaggle/input/...stage2.pth"   # change this
# DATA_ROOT = "/kaggle/input/140k-real-and-fake-faces/real_vs_fake/real-vs-fake"

# test_set = SuperpixelImageFolder(
#     root=os.path.join(DATA_ROOT, "test"),
#     n_segments=25,
#     image_size=224
# )

# test_loader = DataLoader(
#     test_set,
#     batch_size=16,
#     shuffle=False,
#     num_workers=4,
#     pin_memory=True
# )

# model = Stage2(
#     n_segments=25,
#     imagenet_backbone=True
# ).to(device)

# state = torch.load(CKPT_PATH, map_location=device)
# model.load_state_dict(state, strict=True)
# model.eval()

# real_idx = test_set.class_to_idx.get("real", 1)

# te_loss, auc, f1, eer, acc = evaluate_with_metrics(
#     model,
#     test_loader,
#     device,
#     real_idx=real_idx
# )

# print(f"TEST ONLY -> loss={te_loss:.4f} | AUROC={auc:.3f} | F1={f1:.3f} | EER={eer:.3f} | ACC={acc:.3f}")
